# 01 — Pre-Heating Stage Explorer

Hot process air heats the particle bed **before spraying begins**.
No solvent, no coating — only convective heat exchange between gas and particles.

**Move the sliders** to explore how inlet conditions affect:
- The particle heat-up trajectory (T_product)
- The quasi-steady gas temperature (T_gas)
- The driving force for heat transfer (ΔT = T_gas − T_particle)


In [4]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

from fluid_bed.parameters import ProcessParameters
from fluid_bed.models.preheating import run_preheating

%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "font.size": 11})


In [5]:
# Fixed particle and bed properties (typical ethylcellulose-coated pellets)
FIXED = dict(
    particle_density = 1000.0,   # kg/m³
    cp_particle      = 1400.0,   # J/kg/K
    diameter_bed     = 0.60,     # m — pilot-scale coater
    rho_air          = 1.10,     # kg/m³  (warm air ~65 °C)
    cp_air           = 1010.0,   # J/kg/K
)
print("Fixed properties loaded.")


Fixed properties loaded.


In [6]:
S = dict(continuous_update=False)   # shared option: update only on slider release


def _run01(T_inlet_C=70.0, flow_m3h=300.0, T_init_C=20.0,
           duration_min=20.0, diameter_um=200.0, batch_kg=2.0):
    m_air     = (flow_m3h / 3600.0) * FIXED["rho_air"]    # kg/s
    T_inlet_K = T_inlet_C + 273.15
    T_init_K  = T_init_C  + 273.15

    params = ProcessParameters(
        **FIXED,
        diameter_eq      = diameter_um * 1e-6,
        batch_size       = batch_kg,
        air_flow_rates   = (m_air, m_air, m_air),
        air_temperatures = (T_inlet_K, T_inlet_K, T_inlet_K),
    )

    res   = run_preheating(params, duration=duration_min * 60.0, T_particle_init=T_init_K)
    t_min = res.t / 60.0
    T_p_C = res.T_particle - 273.15
    T_g_C = res.T_gas      - 273.15
    dT    = T_g_C - T_p_C

    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    # — Temperature profiles —
    ax = axes[0]
    ax.plot(t_min, T_g_C, "r--", lw=1.5,  label="Gas T_g  (quasi-steady)")
    ax.plot(t_min, T_p_C, "b-",  lw=2.5,  label="Product T_p")
    ax.axhline(T_inlet_C, color="r", ls=":", alpha=0.35,
               label=f"Inlet setpoint {T_inlet_C:.0f} °C")
    ax.set_xlabel("Time (min)"); ax.set_ylabel("Temperature (°C)")
    ax.set_title("Temperature profiles"); ax.legend(); ax.grid(True, alpha=0.3)

    # — Driving force —
    ax = axes[1]
    ax.plot(t_min, dT, "k-", lw=2)
    ax.fill_between(t_min, dT, 0, alpha=0.15, color="orange")
    ax.axhline(0, color="grey", ls="--", alpha=0.5)
    ax.set_xlabel("Time (min)")
    ax.set_ylabel("ΔT = T_gas − T_particle  (°C)")
    ax.set_title("Heat-transfer driving force"); ax.grid(True, alpha=0.3)

    plt.tight_layout(); plt.show()
    print(f"Final product temperature : {T_p_C[-1]:.1f} °C  "
          f"(inlet = {T_inlet_C:.0f} °C)   |   ODE converged: {res.success}")


interact(
    _run01,
    T_inlet_C    = FloatSlider(70.0, min=40,  max=100, step=5,   description="T inlet (°C)",    **S),
    flow_m3h     = FloatSlider(300,  min=100, max=600, step=25,  description="Air flow (m³/h)", **S),
    T_init_C     = FloatSlider(20.0, min=10,  max=35,  step=5,   description="T₀ particle (°C)",**S),
    duration_min = FloatSlider(20.0, min=5,   max=60,  step=5,   description="Duration (min)",  **S),
    diameter_um  = FloatSlider(200,  min=100, max=500, step=50,  description="d_particle (µm)", **S),
    batch_kg     = FloatSlider(2.0,  min=0.5, max=5.0, step=0.5, description="Batch (kg)",      **S),
)


interactive(children=(FloatSlider(value=70.0, continuous_update=False, description='T inlet (°C)', min=40.0, s…

<function __main__._run01(T_inlet_C=70.0, flow_m3h=300.0, T_init_C=20.0, duration_min=20.0, diameter_um=200.0, batch_kg=2.0)>